# Product Category Classification

Notebook pentru analiza, feature engineering, antrenarea și evaluarea modelului.

## 1. Importuri

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import re

## 2. Încărcarea datelor

In [ ]:
df = pd.read_csv("../data/products.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

Observație: unele coloane au spații la început sau la final, deci curățăm numele coloanelor.

In [ ]:
df.columns = df.columns.str.strip()
df.columns

## 3. Analiza valorilor lipsă

In [ ]:
df.isna().sum()

In [ ]:
df = df.dropna(subset=["Product Title", "Category Label"]).copy()
df["Product Title"] = df["Product Title"].astype(str).str.lower().str.strip()
df["Category Label"] = df["Category Label"].astype(str).str.strip()
df.shape

## 4. Distribuția categoriilor

In [ ]:
df["Category Label"].value_counts()

In [ ]:
plt.figure(figsize=(10, 6))
df["Category Label"].value_counts().plot(kind="bar")
plt.title("Distribuția produselor pe categorii")
plt.xlabel("Categorie")
plt.ylabel("Număr produse")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Feature engineering

Creăm câteva caracteristici exploratorii pentru titlu.

In [ ]:
df["title_length"] = df["Product Title"].apply(len)
df["word_count"] = df["Product Title"].apply(lambda x: len(x.split()))
df["has_numbers"] = df["Product Title"].apply(lambda x: int(any(char.isdigit() for char in x)))
df[["Product Title", "title_length", "word_count", "has_numbers"]].head()

Pentru modelul final folosim TF-IDF pe cuvinte și pe caractere. N-gramurile pe caractere sunt utile deoarece titlurile conțin multe coduri de produs.

In [ ]:
def category_tokens(category):
    tokens = set(re.findall(r"[a-z]+", category.lower()))
    tokens.update({token[:-1] for token in tokens if token.endswith("s") and len(token) > 3})
    return tokens


def augment_titles(data):
    rows = []
    for _, row in data.iterrows():
        title = row["Product Title"]
        category = row["Category Label"]
        tokens = category_tokens(category)
        if not tokens:
            continue
        pattern = r"\b(" + "|".join(map(re.escape, sorted(tokens, key=len, reverse=True))) + r")\b"
        cleaned_title = re.sub(pattern, " ", title)
        cleaned_title = re.sub(r"\s+", " ", cleaned_title).strip()
        if cleaned_title and cleaned_title != title and len(cleaned_title) >= 5:
            rows.append({"Product Title": cleaned_title, "Category Label": category})
    return pd.concat([data, pd.DataFrame(rows)], ignore_index=True)

augmented_df = augment_titles(df[["Product Title", "Category Label"]])
augmented_df.shape

## 6. Train-test split

In [ ]:
X = df["Product Title"]
y = df["Category Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_df = pd.DataFrame({"Product Title": X_train, "Category Label": y_train})
train_augmented = augment_titles(train_df)
train_augmented.shape

## 7. Model 1 – Linear SVM

In [ ]:
svm_model = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=12000, min_df=2)),
        ("char_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=12000, min_df=2)),
    ])),
    ("classifier", LinearSVC(random_state=42))
])

svm_model.fit(train_augmented["Product Title"], train_augmented["Category Label"])
svm_preds = svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_preds)
svm_accuracy

## 8. Model 2 – Multinomial Naive Bayes

In [ ]:
nb_model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=12000, min_df=2)),
    ("classifier", MultinomialNB())
])

nb_model.fit(train_augmented["Product Title"], train_augmented["Category Label"])
nb_preds = nb_model.predict(X_test)
nb_accuracy = accuracy_score(y_test, nb_preds)
nb_accuracy

## 9. Evaluarea modelului final

In [ ]:
print("Linear SVM accuracy:", svm_accuracy)
print("Naive Bayes accuracy:", nb_accuracy)
print(classification_report(y_test, svm_preds, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, svm_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="Blues")
plt.title("Confusion Matrix - Linear SVM")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 10. Salvarea modelului final

In [ ]:
final_model = svm_model
joblib.dump(final_model, "../models/product_classifier.pkl")

## 11. Testare manuală

In [ ]:
examples = [
    "iphone 7 32gb gold",
    "olympus e m10 mark iii geh use silber",
    "kenwood k20mss15 solo",
    "bosch wap28390gb 8kg 1400 spin",
    "bosch serie 4 kgv39vl31g",
    "smeg sbs8004po",
]

for title in examples:
    print(title, "->", final_model.predict([title])[0])

## 12. Raport generat de script

```text
Product Category Classification Report
=============================================
Rows used after cleaning: 35096
Rows after title augmentation for final training: 52565
Number of categories: 13

Model comparison on held-out test set:
- Linear SVM: accuracy=0.9811
- Multinomial Naive Bayes: accuracy=0.9387

Best model: Linear SVM
Best accuracy: 0.9811

Classification report for best model:
                  precision    recall  f1-score   support

             CPU       0.00      0.00      0.00        17
            CPUs       0.98      1.00      0.99       749
 Digital Cameras       1.00      0.99      0.99       538
     Dishwashers       0.99      1.00      0.99       681
        Freezers       0.99      0.96      0.98       440
 Fridge Freezers       0.98      0.99      0.98      1094
         Fridges       0.94      0.97      0.95       687
      Microwaves       0.98      1.00      0.99       466
    Mobile Phone       0.00      0.00      0.00        11
   Mobile Phones       0.97      1.00      0.98       801
             TVs       1.00      0.99      1.00       708
Washing Machines       1.00      0.98      0.99       803
          fridge       0.00      0.00      0.00        25

        accuracy                           0.98      7020
       macro avg       0.76      0.76      0.76      7020
    weighted avg       0.97      0.98      0.98      7020


Manual test examples using final saved model:
- iphone 7 32gb gold -> Mobile Phones
- olympus e m10 mark iii geh use silber -> Digital Cameras
- kenwood k20mss15 solo -> Microwaves
- bosch wap28390gb 8kg 1400 spin -> Washing Machines
- bosch serie 4 kgv39vl31g -> Fridge Freezers
- smeg sbs8004po -> Fridge Freezers
```